In [1]:
import os
import sys

import logging

%pwd
os.chdir("../")

# Add the absolute path to src/ so Python can find automatch
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.append(src_path)
    
%pwd


from src.onBench.utils.EDA import *


logging.getLogger("snowflake").setLevel(logging.WARNING)
logging.getLogger("snowflake.connector").setLevel(logging.WARNING)
logging.getLogger("snowflake.snowpark").setLevel(logging.WARNING)





c:\Users\fiscarelli\Desktop\Progetti\Manpower IT\OnBench\On-Bench-Candidates-to-Jobs-Auto-Match\venv\lib\site-packages\holoviews\__init__.py:81: FutureWarning: param.version.Version has been deprecated and will be removed in a future version.
  __version__ = str(param.version.Version(fpath=__file__, archive_commit="$Format:%h$",
c:\Users\fiscarelli\Desktop\Progetti\Manpower IT\OnBench\On-Bench-Candidates-to-Jobs-Auto-Match\venv\lib\site-packages\param\version.py:200: FutureWarning: param.version.run_cmd has been deprecated and will be removed in a future version.
  remotes = run_cmd([cmd, 'remote', '-v'],
c:\Users\fiscarelli\Desktop\Progetti\Manpower IT\OnBench\On-Bench-Candidates-to-Jobs-Auto-Match\venv\lib\site-packages\panel\config.py:35: FutureWarning: param.version.Version has been deprecated and will be removed in a future version.
  __version__ = str(param.version.Version(
c:\Users\fiscarelli\Desktop\Progetti\Manpower IT\OnBench\On-Bench-Candidates-to-Jobs-Auto-Match\venv\lib\si

In [2]:
from src.onBench.utils.snowflake_utils import get_snowpark_session
session = get_snowpark_session()

In [3]:
df = session.sql(f"""select *
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_CANDIDATE_CLEANED
                     --where status = 'No mission'
                 """).to_pandas()


In [4]:
df["CCNL_CODE"] = df["CCNL_CODE"].astype("Int64").astype("string")

df[df.select_dtypes(include=["object", "string"]).columns] = (
    df.select_dtypes(include=["object", "string"]).apply(lambda col: col.str.strip())
)


In [19]:
keywords = [

    #general
    "curriculum", "cv", "vitae", "esperienza", "lavoro", "occupazione", "posizione", "ruolo", "azienda",
    # ruoli
    "addetto", "operaio", "impiegato", "responsabile", "cassiere", "magazziniere",
    "analista", "developer", "autista", "segretaria", "tecnico", "consulente",

    # titoli di studio
    "diploma", "laurea", "master", "istituto", "università",

    # competenze
    "excel", "office", "inglese", "informatica", "patente",

    # pattern temporali
    #"2019", "2020", "2021", "2022", "2023", "2024", "dal", "al",

    # aziende / enti
    "srl", "spa", "cooperativa", "comune", "regione"
]

df["IS_VALID_CV"] = df["CURRICULUM"].str.lower().apply(
    lambda x: any(k in x for k in keywords) if isinstance(x, str) else False
)

is_valid_cv = np.sum(df["IS_VALID_CV"])/df.shape[0]

print(f"""Percentage of invalid CVs (custom keywords matching): {1-is_valid_cv:.2%}""")




Percentage of invalid CVs (custom keywords matching): 50.48%


In [ ]:
general_info(df)


Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 416 entries, 0 to 415
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   CANDIDATEID               416 non-null    int32         
 1   SALARY                    5 non-null      float64       
 2   DATE_AVAILABLE            352 non-null    datetime64[ns]
 3   WILL_RELOCATE             416 non-null    bool          
 4   DBJOBCANDNO               416 non-null    int32         
 5   START_DATE                416 non-null    datetime64[ns]
 6   END_DATE                  416 non-null    datetime64[ns]
 7   PARTTIME_PREFERENZA_PERC  416 non-null    int8          
 8   TURNO_PREFERENZA          2 non-null      object        
 9   PARTTIME_PERC             416 non-null    float64       
 10  PB_ID                     416 non-null    object        
 11  FIRST_NAME                416 non-null    object        
 12  LAST_NAME      

Features that we can use in the re-ranker:
- PARTTIME_PERC: Part-time/Full-time (primo contratto)
- CCNL_CODE, CCNL_DESCRIPTION: Primo CCNL (primo contratto)
- CCNL_LEVEL: Livello Primo CCNL (primo contratto)
- LEGAL_CATEGORY: Categoria Legale (primo contratto)

Columns with a lot of missing data:
-   salary (99% missing)
-   turno_preferenza (99.5% missing)
-   skills (78% missing)
-   education (71% missing)
-   certifications (100% missing)
-   curriculum 38% missing
-   description (26% missing. this means that curriculum, skills, education and certifiations are all simultaneaously missing)

In [ ]:
columns_to_check = ["PARTTIME_PREFERENZA_PERC", "PARTTIME_PERC", "EDUCATION", "CERTIFICATIONS"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)


Unique values in 'PARTTIME_PREFERENZA_PERC':
[100]
----------------------------------------
Unique values in 'PARTTIME_PERC':
[ 10.    12.5   17.78  20.    25.    25.9   37.5   38.46  50.    51.28
  60.    61.53  62.5   75.    76.93  80.    87.5  100.  ]
----------------------------------------
Unique values in 'EDUCATION':
['bachelor' 'diploma' 'laurea' 'laurea breve' 'non specificato'
 'scuola media' 'scuola secondaria' 'universita' None]
----------------------------------------
Unique values in 'CERTIFICATIONS':
[None]
----------------------------------------
Unique values in 'SKILLS':
[''
 '2d,3d,adobe photoshop,autodesk,autodesk inventor,cad/cam &gt; cad,creo,ios,microsoft dos,microsoft windows,openoffice,rhinoceros,solidworks,ubuntu,,/'
 '2d,3d,attestato di partecipazione,autocad,autodesk,bracciante agricolo,chartered asset manager,dec alpha hardware,diploma,english,italian,tecnico'
 '2d,autocad,microsoft excel,microsoft excel 2000,microsoft powerpoint,microsoft word,microsoft wo

In [8]:
columns_to_check = ["CCNL_LEVEL", "CCNL_CODE", "LEGAL_CATEGORY", "CCNL_DESCRIPTION"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)



Unique values in 'CCNL_LEVEL':
['1' '2' '3' '3/1' '4' '4 junior' '4 senior' '5' '5 fascia 1' '6'
 '6 junior' '6 senior' '6 super (ex 7)' '7' '8' 'a' 'a3' 'ae1' 'ap prima'
 'area 3 liv. 1' 'b' 'b/1' 'b1' 'b2' 'b3' 'c' 'c/1' 'c/2' 'c1' 'c2' 'c3'
 'd/2' 'd1' 'd2' 'd3' 'e' 'e1' 'e2' 'e3' 'e4' 'g' 'g1' 'h' 'i' 'iii'
 'op centrato' 'op consolidato' 'operat esperti' 'v']
----------------------------------------
Unique values in 'CCNL_CODE':
<StringArray>
['1010',  '106', '1220', '1270',  '130', '1380', '1390', '1790', '1810',
 '1820', '1870', '1920', '1930', '1980',   '20', '2070', '2220',  '230',
 '2310', '2360',  '240',  '250', '2560',  '330', '3420',   '38',   '39',
  '530',  '710',  '720',   '81', '9013', '9332', '9335', '9354', '9408',
 '9504', '9631', '9646', '9662', '9669', '9674', '9683', '9719', '9726',
 '9746', '9763', '9776', '9791', '9796', '9816', '9817',   <NA>]
Length: 53, dtype: string
----------------------------------------
Unique values in 'LEGAL_CATEGORY':
['impiegati' 'op

In [25]:
b1 = bar_plot(df, 'EDUCATION', rotation=25)
b2 = bar_plot(df, 'CERTIFICATIONS', rotation=25)
b3 = bar_plot(df, 'IS_VALID_CV', rotation=25)

(b1 + b2 + b3).cols(1)

:Layout
   .Bars.I   :Bars   [EDUCATION]   (COUNT)
   .Bars.II  :Bars   [CERTIFICATIONS]   (COUNT)
   .Bars.III :Bars   [IS_VALID_CV]   (COUNT)

In [26]:
b1 = bar_plot(df, 'CCNL_LEVEL')#, label_range=30)
b2 = bar_plot(df, 'CCNL_CODE')#, label_range=30)
b3 = bar_plot(df, 'LEGAL_CATEGORY')#, label_range=30)
b4 = bar_plot(df, 'CCNL_DESCRIPTION', rotation=25)

(b1 + b2 + b3 + b4).cols(1)

:Layout
   .Bars.I   :Bars   [CCNL_LEVEL]   (COUNT)
   .Bars.II  :Bars   [CCNL_CODE]   (COUNT)
   .Bars.III :Bars   [LEGAL_CATEGORY]   (COUNT)
   .Bars.IV  :Bars   [CCNL_DESCRIPTION]   (COUNT)

In [27]:
b1 = bar_plot(df, 'JOB_FIRST', rotation=25)#, label_range=30)
b2 = bar_plot(df, 'JOB_LAST', rotation=25)

(b1 + b2).cols(1)

:Layout
   .Bars.I  :Bars   [JOB_FIRST]   (COUNT)
   .Bars.II :Bars   [JOB_LAST]   (COUNT)

In [12]:
b1 = bar_plot(df, 'JOB_SITE', rotation=25)#, label_range=30)
b1

:Bars   [JOB_SITE]   (COUNT)

In [13]:
h1 = hist_plot(df, "PARTTIME_PERC", bins=20, xrange=[], yrange=[], height=400, width=800)

h1

:Histogram   [PARTTIME_PERC]   (Frequency)

# VACANCIES